In [2]:
import onnxruntime as ort
import numpy as np
import scipy.special
from PIL import Image

def preprocess_image(image, resize_size=256, crop_size=224, mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]):
  image = image.resize((resize_size, resize_size), Image.BILINEAR)
  w, h = image.size
  left = (w - crop_size) / 2
  top = (h - crop_size) / 2
  image = image.crop((left, top, left + crop_size, top + crop_size))
  image = np.array(image).astype(np.float32)
  image = image / 255.0
  image = (image - mean) / std
  image = np.transpose(image, (2, 0, 1))
  image = image.reshape((1,) + image.shape)
  return image

session = ort.InferenceSession('resnet.onnx')

with open('labels.txt') as f:
    labels = [line.strip() for line in f.readlines()]

input_name = session.get_inputs()[0].name
output_name = session.get_outputs()[0].name

image = Image.open('img_test.jpg').convert('RGB')
processed_image = preprocess_image(image)
processed_image = processed_image.astype(np.float32)

output = session.run([output_name], {input_name: processed_image})[0]

probabilities = scipy.special.softmax(output, axis=-1)

# 获取最高的5个概率和对应的类别索引
top5_idx = np.argsort(probabilities[0])[-5:][::-1]
top5_prob = probabilities[0][top5_idx]

for i in range(5):

  print(f"{i + 1}: {labels[top5_idx[i]]} - Probability: {top5_prob[i]}")



1: tusker - Probability: 0.931233286857605
2: Indian elephant - Probability: 0.06785939633846283
3: African elephant - Probability: 0.0009034110116772354
4: gorilla - Probability: 4.78091294553451e-07
5: water buffalo - Probability: 3.9817274455344887e-07
